# Data Extraction: Customer Lifetime Value Dataset

This notebook extracts customer features from Google BigQuery's [TheLook E-Commerce](https://console.cloud.google.com/marketplace/product/bigquery-public-data/thelook-ecommerce) public dataset for building a Customer Lifetime Value (CLV) model using the BG/NBD + Gamma-Gamma probabilistic framework.

**Pipeline:**
1. Connect to BigQuery with cost controls
2. Execute parameterized CLV feature engineering SQL
3. Validate data quality (BG/NBD assumptions)
4. Export to CSV for modeling in `02_purchase_propensity.ipynb`

In [1]:
import pandas as pd
import os
from google.cloud import bigquery
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Config
PROJECT_ID = "churn-portfolio-project"
SQL_FILE_PATH = "../sql/clv_features.sql"
DATA_PATH = "../data/raw/clv_data.csv"

# Parameters
CUTOFF_DATE = "2025-10-01"       # End of holdout window (simulated "today")
CALIBRATION_END = "2025-04-04"   # End of calibration period (cutoff - 180 days)

## CLV Modeling Framework

We use a **temporal train/holdout split** to validate the model before deployment:

| Period | Dates | Purpose |
|--------|-------|---------|
| Calibration | Before 2025-04-04 | Fit BG/NBD + Gamma-Gamma models |
| Holdout | 2025-04-04 → 2025-10-01 (180 days) | Validate CLV predictions |

**BG/NBD Inputs (computed at calibration end):**

| Feature | Definition | Notes |
|---------|------------|-------|
| `frequency` | Repeat purchases (total_orders - 1) | 0 = one-time buyer (~88% of customers) |
| `recency` | Days from first to last purchase | 0 for one-time buyers |
| `T` | Customer age in days at calibration end | Must be > 0 |
| `monetary_value` | Avg revenue per repeat transaction | Fallback to AOV for one-time buyers |

**Design Decision:** One-time buyers are NOT excluded — BG/NBD handles them natively and their low CLV output is the correct, informative answer.

## Feature Engineering

Features are extracted from three source tables:

### BG/NBD Core Inputs
*Source: `order_items` (calibration period)*

| Feature | Description |
|---------|-------------|
| `frequency` | Number of repeat purchases (total_orders - 1) |
| `recency` | Days from first to last purchase within calibration |
| `T` | Customer age at calibration end (days since first purchase) |
| `monetary_value` | Avg order value on repeat transactions |
| `avg_order_value` | Overall avg order value (context + fallback) |

### Holdout Labels
*Source: `order_items` (holdout period)*

| Feature | Description |
|---------|-------------|
| `actual_holdout_transactions` | Orders placed in holdout window |
| `actual_holdout_revenue` | Revenue generated in holdout window |

### Demographic Features
*Source: `users`*

| Feature | Description |
|---------|-------------|
| `customer_tenure_days` | Days since account creation |
| `age`, `gender` | Demographics |
| `traffic_source` | Acquisition channel |
| `country` | Geographic market |

### Engagement Features
*Source: `events`*

| Feature | Description |
|---------|-------------|
| `total_sessions` | Site visit count |
| `days_since_last_visit` | Browsing recency |
| `cart_events`, `product_view_events` | Purchase intent signals |

In [3]:
# Load and display SQL query
with open(SQL_FILE_PATH) as f:
    query = f.read()

# Show first 50 lines for reference
print("\n".join(query.split("\n")[:50]))
print("\n... [truncated] ...")

/*
  CLV Feature Engineering: Customer-Level Purchase History + Engagement Features

  Computes customer-level features for Customer Lifetime Value (CLV) prediction
  using a two-stage model: (1) purchase propensity classifier, (2) spend-tier
  expected revenue, combined as CLV = P(purchase) x E[revenue | purchase].

  Data Sources:
    - order_items: Transaction history (purchase features + holdout labels)
    - users:       Demographics and acquisition channel
    - events:      Behavioral engagement signals

  Parameters:
    @cutoff_date:       End of holdout window (simulated "today")    → 2025-10-01
    @calibration_end:   End of calibration period (model training cutoff) → 2025-04-04
                        (cutoff_date minus 180 days)

  Purchase History Features (computed at calibration_end):
    frequency      = total distinct orders - 1   (repeat purchases; 0 = one-time buyer)
    recency        = days from first to last purchase within calibration period
    T              

## Query Execution

In [4]:
# Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Set up query job configuration with parameters
job_config = bigquery.QueryJobConfig(
    maximum_bytes_billed=500 * 1024**2,  # 500MB safety cap
    query_parameters=[
        bigquery.ScalarQueryParameter("cutoff_date",      "DATE", CUTOFF_DATE),
        bigquery.ScalarQueryParameter("calibration_end",  "DATE", CALIBRATION_END),
    ],
)

In [5]:
# Dry run: Estimate cost before execution
job_config.dry_run = True
job = client.query(query, job_config=job_config)

bytes_processed = job.total_bytes_processed
mb = bytes_processed / 1e6
cost = bytes_processed / 1e12 * 5  # $5 per TB

print(f"Estimated data scan: {mb:.2f} MB")
print(f"Estimated cost: ${cost:.4f}")

Estimated data scan: 154.35 MB
Estimated cost: $0.0008


In [6]:
# Execute query (or load cached data)
if os.path.exists(DATA_PATH):
    print(f"Loading cached data from {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    if mb > 500:
        raise ValueError(f"Query too large ({mb:.0f} MB). Adjust parameters.")

    print("Executing BigQuery...")
    job_config.dry_run = False
    df = client.query(query, job_config=job_config).to_dataframe()

    # Cache locally
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    df.to_csv(DATA_PATH, index=False)
    print(f"Saved to {DATA_PATH}")

print(f"\nDataset shape: {df.shape[0]:,} customers × {df.shape[1]} features")

Loading cached data from ../data/raw/clv_data.csv

Dataset shape: 52,507 customers × 24 features


## Data Validation

In [7]:
# Schema check
print("=== Schema ===")
print(df.dtypes)
print(f"\n=== Missing Values ===")
print(df.isna().sum()[df.isna().sum() > 0] if df.isna().sum().sum() > 0 else "No missing values")

=== Schema ===
user_id                          int64
frequency                        int64
recency                          int64
T                                int64
monetary_value                 float64
total_orders                     int64
total_spend                    float64
avg_order_value                float64
days_since_last_order            int64
actual_holdout_transactions      int64
actual_holdout_revenue         float64
customer_tenure_days             int64
age                              int64
gender                          object
traffic_source                  object
country                         object
total_sessions                   int64
total_events                     int64
days_since_last_visit            int64
avg_events_per_session         float64
distinct_event_types             int64
cart_events                      int64
product_view_events              int64
purchase_events                  int64
dtype: object

=== Missing Values ===
No missing 

In [8]:
# BG/NBD data quality assertions
assert df['user_id'].is_unique,                        "Duplicate user IDs found"
assert (df['frequency'] >= 0).all(),                   "frequency must be non-negative"
assert (df['recency'] <= df['T']).all(),               "recency cannot exceed T"
assert (df['T'] > 0).all(),                            "T must be positive (T > 7 filter)"
assert (df['monetary_value'] > 0).all(),               "monetary_value must be positive for Gamma-Gamma"
assert df['frequency'].dtype in ['int64', 'Int64'],    "frequency should be integer"

print("✓ All BG/NBD data quality checks passed")

✓ All BG/NBD data quality checks passed


In [ ]:
# One-time buyer rate
one_time_rate = (df['frequency'] == 0).mean()
print(f"=== Purchase Frequency Distribution ===")
print(f"One-time buyers (frequency=0): {(df['frequency']==0).sum():,} ({one_time_rate:.1%})")
print(f"Repeat buyers (frequency≥1):   {(df['frequency']>=1).sum():,} ({1-one_time_rate:.1%})")
print(f"\nFrequency stats:")
print(df['frequency'].describe())

=== Purchase Frequency Distribution ===
One-time buyers (frequency=0): 36,354 (69.2%)
Repeat buyers (frequency≥1):   16,153 (30.8%)

Frequency stats:
count    52507.000000
mean         0.422896
std          0.726733
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          3.000000
Name: frequency, dtype: float64


In [10]:
# BG/NBD input summary
bgnbd_cols = ['frequency', 'recency', 'T', 'monetary_value']
print("=== BG/NBD Core Inputs ===")
df[bgnbd_cols].describe().round(2)

=== BG/NBD Core Inputs ===


,frequency,recency,T,monetary_value
count,52507.00,52507.00,52507.00,52507.00
mean,0.42,165.32,723.39,86.55
std,0.73,346.13,532.09,91.82
min,0.00,0.00,8.00,0.02
25%,0.00,0.00,275.00,30.00
50%,0.00,0.00,615.00,58.00
75%,1.00,132.00,1092.00,109.95
max,3.00,2164.00,2276.00,1264.60


In [11]:
# Holdout label summary
holdout_buyers = (df['actual_holdout_transactions'] > 0).sum()
print(f"=== Holdout Period (180 days) ===")
print(f"Customers who purchased in holdout: {holdout_buyers:,} ({holdout_buyers/len(df):.1%})")
print(f"Total holdout transactions:         {df['actual_holdout_transactions'].sum():,}")
print(f"Total holdout revenue:              ${df['actual_holdout_revenue'].sum():,.2f}")

# Categorical feature summary
categorical_cols = ['gender', 'traffic_source', 'country']
for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(df[col].value_counts().head(5))

=== Holdout Period (180 days) ===
Customers who purchased in holdout: 6,451 (12.3%)
Total holdout transactions:         7,221
Total holdout revenue:              $612,372.39

=== gender ===
gender
F    26320
M    26187
Name: count, dtype: int64

=== traffic_source ===
traffic_source
Search      36853
Organic      7918
Facebook     3127
Email        2509
Display      2100
Name: count, dtype: int64

=== country ===
country
China            17843
United States    11627
Brasil            7807
South Korea       2816
France            2524
Name: count, dtype: int64


In [12]:
# Sample records
df.head()

,user_id,frequency,recency,T,monetary_value,total_orders,total_spend,avg_order_value,days_since_last_order,actual_holdout_transactions,...,traffic_source,country,total_sessions,total_events,days_since_last_visit,avg_events_per_session,distinct_event_types,cart_events,product_view_events,purchase_events
0,34756,1,993,1064,411.750000,2,465.490002,232.745001,71,0,...,Search,China,7,82,70,11.714286,4,25,25,7
1,9513,2,743,1062,94.719997,3,297.019993,99.006664,319,0,...,Search,China,6,49,319,8.166667,5,14,14,6
2,74483,2,436,685,150.335000,3,321.920000,107.306667,249,1,...,Organic,South Korea,7,71,249,10.142857,5,21,21,7
3,69957,1,138,1701,196.759998,2,239.569998,119.784999,1563,0,...,Search,France,7,82,1559,11.714286,4,25,25,7
4,67994,2,362,1576,156.990001,3,435.670000,145.223333,1214,0,...,Facebook,China,8,80,1214,10.000000,4,24,24,8


---

**Next:** [02_purchase_propensity.ipynb](./02_purchase_propensity.ipynb)